# rbf-designer — LoRA ince ayar (Colab)

Uzman gosterimleriyle (`sft.jsonl`) bir taban modeli ince ayar eder; sonuc
devre fizigini AGIRLIKLARINDA tasiyan, Cadence araclarini surebilen kendi
modelinizdir. Cikti Ollama'ya alinabilen GGUF olur.

**ONCE:** Runtime -> Change runtime type -> **T4 GPU**. Sonra **Runtime -> Run all**
(hucreleri bastan sirayla koşar — atlamayin). Bir hucrede hata cikarsa durun,
o hucrenin gorüntüsünü paylasin.


## 1) GPU kontrolu


In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU yok! Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))


## 2) Kurulum (~2-3 dk)


In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets


## 3) sft.jsonl yukle

Yerelde `python finetune.py dataset --n-tasks 300 --out sft.jsonl` ile urettiginiz
dosyayi asagidaki dugmeyle secin (C:\\Users\\ozgur\\rbf-gpt-win\\sft.jsonl).


In [ ]:
from google.colab import files
up = files.upload()   # sft.jsonl secin
DATA = next(iter(up)) if up else 'sft.jsonl'
import json
n = sum(1 for _ in open(DATA))
print('veri:', DATA, '|', n, 'gorev')


## 4a) Taban modeli yukle (Qwen2.5-7B, 4-bit)


In [ ]:
from unsloth import FastLanguageModel
MAXLEN = 2048   # evirici diyaloglari kisa; T4 bellegi icin 2048 yeterli/guvenli
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-7B-Instruct-bnb-4bit',
    max_seq_length=MAXLEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    use_gradient_checkpointing='unsloth',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'])


## 4b) Veriyi sohbet bicimine cevir

Her ornek system+user+arac-cagrilarindan olusuyor; Qwen sablonu bunlari tek
egitim metnine cevirir. Sablon arac-cagrilarini render edemezse anlasilir bir
hata veririz (bu durumda gorüntüyü paylasin).


In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files=DATA, split='train')
def bicim(e):
    return {'text': tok.apply_chat_template(e['messages'], tokenize=False)}
try:
    ds = ds.map(bicim, remove_columns=ds.column_names)
except Exception as ex:
    raise RuntimeError('Chat sablonu arac-cagrilarini render edemedi: '
                       + repr(ex))
print('ornek sayisi:', len(ds))
print(ds[0]['text'][:600])


## 4c) Egitim (~10-15 dk, T4)


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAXLEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=10, optim='adamw_8bit', seed=0,
        output_dir='rbf-designer-lora', report_to='none'))
trainer.train()


## 5) Hizli test — model orani degistiriyor mu?

Egitimden sonra model, Vm'i hedefe tasimak icin ORANI gezmeli (taban modelin
yapamadigi sey). measure_inverter cagrisi uretmesini bekliyoruz.


In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {'role':'system','content':'Sen bir analog devre tasarim ajanisin. '
     'measure_inverter(wn_nm, wp_nm) araci Vm doner. Vm sadece Wp/Wn oranina '
     'baglidir; orani buyutmek Vm''i yukseltir. Hedefe ulasmak icin orani ayarla.'},
    {'role':'user','content':'Eviriciyi Vm=0.6V olacak sekilde boyutlandir.'}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              return_tensors='pt').to('cuda')
out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.3)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))


## 6) Modeli kaydet + GGUF (Ollama icin)

Once LoRA'yi taban modele birlestirip 16-bit kaydederiz (guvence: GGUF adimi
takilsa bile model elde kalir), sonra q4_k_m GGUF uretiriz.


In [ ]:
# Guvence: once birlesik 16-bit HF modeli
model.save_pretrained_merged('rbf-designer-merged', tok,
                             save_method='merged_16bit')
print('birlesik model kaydedildi: rbf-designer-merged/')


In [ ]:
# GGUF (Ollama). Bu adim llama.cpp derledigi icin birkac dk surebilir.
model.save_pretrained_gguf('rbf-designer', tok,
                           quantization_method='q4_k_m')
!ls -lh rbf-designer/*.gguf


## 7) Indir ve Ollama'ya al

Asagidaki hucre GGUF'u ve Modelfile'i zip'leyip indirir. Windows'ta zip'i acip
klasore girin:

```powershell
ollama create rbf-designer -f Modelfile
python llm_controller.py --task "Eviriciyi Vm=0.6V olacak sekilde boyutlandir" --llm rbf-designer
```

Not: `ollama show rbf-designer --modelfile` ciktisinda `FROM ./rbf-designer...gguf`
gorurseniz artik EGITILMIS modeli calistiriyorsunuz demektir (prompt surumu degil).


In [ ]:
import glob, os
gguf = sorted(glob.glob('rbf-designer/*.gguf'))[0]
ad = os.path.basename(gguf)
open('rbf-designer/Modelfile','w').write(
    f'FROM ./{ad}\nPARAMETER temperature 0.3\n')
!cd rbf-designer && zip -r ../rbf-designer.zip *.gguf Modelfile
from google.colab import files
files.download('rbf-designer.zip')
